# Evaluation and Metrics
This notebook evaluates model outputs and summarizes scenario performance, battery benefits, and shortfall reduction.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

project_root = Path.cwd()
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

metrics_path = processed_dir / 'battery_storage_analysis.csv'
recommendations_path = processed_dir / 'battery_recommendations.csv'
scenarios_path = processed_dir / 'scenario_forecasts.csv'

metrics_df = pd.read_csv(metrics_path)
recommendations_df = pd.read_csv(recommendations_path)
scenarios_df = pd.read_csv(scenarios_path)
scenarios_df['date'] = pd.to_datetime(scenarios_df['date'])

metrics_df.head()

In [ ]:
summary = metrics_df.groupby('scenario').agg({
    'shortfall_reduction_pct': 'max',
    'residual_shortfall_gwh': 'min',
    'unmet_days': 'min',
}).reset_index()
summary.rename(columns={
    'shortfall_reduction_pct': 'max_shortfall_reduction_pct',
    'residual_shortfall_gwh': 'min_residual_shortfall_gwh',
    'unmet_days': 'min_unmet_days',
}, inplace=True)
summary_path = processed_dir / 'evaluation_summary.csv'
summary.to_csv(summary_path, index=False)
print(f"Saved evaluation summary to {summary_path}")
summary

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=metrics_df, x='scenario', y='shortfall_reduction_pct', hue='battery_config')
plt.title('Shortfall Reduction by Scenario and Battery Size')
plt.ylabel('Reduction (%)')
plt.xlabel('Scenario')
plt.legend(title='Battery Config')
plt.tight_layout()
plt.show()